# Colab — Image baseline ResNet-18 (MSC RQ1 transfer)

Transfer-learns ResNet-18 on the frozen `fake_news_final_image.tsv` cohort. Same splits as the text baselines: Fakeddit official + FNN 80/20, seed **42**. **A T4 GPU is required** (CPU is ~10–50× slower).

Full cohort ~41k train rows, 3 epochs (~4–8 h on a T4 with AMP) — plan an overnight run; checkpoints save to `runs/` each epoch (free Colab may disconnect after ~12 h or when idle). Outputs → `runs/image_resnet18_baseline/` and `My Drive/runs/`. Setup steps are in the cell below.

## How to run this notebook (Google Colab)

**What it does:** trains and evaluates the **ResNet-18 image baseline**. Needs a **GPU runtime** (a free T4 is sufficient).

**Before you run:**
1. **Add the shared project folder to your Google Drive** (one-time) so these paths exist — the setup cell reads them:
   - `My Drive/training/src/` — helper modules (`colab_setup.py`, `cohort_*.py`)
   - `My Drive/data/fake_news_final_image.tsv` — the frozen image cohort (~8.5 MB)
   - `My Drive/data/images.zip` — the cohort images (~1.3 GB). The setup cell extracts these automatically; the **first run takes a few minutes** to copy and unzip.

   If the folder was *shared with you*, open it in Drive and click **“Add shortcut to Drive”** so it appears under `My Drive/`.
2. **Set the runtime:** `Runtime → Change runtime type → T4 GPU`.
3. **(Recommended)** `File → Save a copy in Drive` so your edits and outputs persist.

**Then `Runtime → Run all`** (or run the cells top to bottom in order):
- **Bootstrap** — mounts Drive and copies `training/` into the session.
- **Dependencies** — `install_dependencies(["image"])` installs the pinned libraries.
- **Setup** — `require_cuda()` checks the GPU, copies the TSV, and extracts the images.
- The remaining cells train and evaluate, saving to `runs/image_resnet18_baseline/` (and persisting it to `My Drive/runs/`).

> **Troubleshooting**
> - `No module named 'google'` → you’re not on a hosted Colab runtime. Open this in the Colab browser tab and connect to a **hosted** runtime (not “local”).
> - `No module named 'colab_setup'` → the **Bootstrap** cell hasn’t run successfully yet, or `My Drive/training/src/` is missing. Run Bootstrap first and confirm the Drive layout above.
> - GPU error from `require_cuda()` → set the runtime to **T4 GPU** (step 2), then `Runtime → Restart and run all`.

In [ ]:
# Bootstrap — mount Drive and copy training/ (identical cell in all Colab notebooks)
import shutil
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/msc")
TRAINING_SRC = Path("/content/drive/MyDrive/training")
TRAINING = PROJECT_ROOT / "training"
if not TRAINING_SRC.is_dir():
    raise FileNotFoundError(
        "Sync repo training/ to My Drive/training/ (modules under training/src/)"
    )
if TRAINING.exists():
    shutil.rmtree(TRAINING)
shutil.copytree(TRAINING_SRC, TRAINING)
sys.path.insert(0, str(TRAINING / "src"))
print("PROJECT_ROOT:", PROJECT_ROOT, "| training:", TRAINING)

In [ ]:
from colab_setup import install_dependencies

# Pinned, version-synced installs (single source of truth: colab_setup.PINNED_DEPENDENCIES).
# torch/torchvision are left to Colab's preinstalled, CUDA-matched build.
install_dependencies(["image"])

In [ ]:
from colab_setup import require_cuda, setup_colab_project

require_cuda()
ctx = setup_colab_project(
    tsv_names=["fake_news_final_image.tsv"],
    need_images=True,
)
PROJECT_ROOT = ctx.project_root


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "training" / "src"))

import pandas as pd
from cohort_image import (
    load_image_cohort,
    train_val_frames,
    verify_image_files,
    verify_jpeg_payloads,
)

df_img = load_image_cohort(PROJECT_ROOT)
_, stats = verify_image_files(df_img, PROJECT_ROOT)
print("Image file check:", stats)

jpeg_stats = verify_jpeg_payloads(df_img, PROJECT_ROOT)
print("JPEG payload check:", jpeg_stats)

train_full, val_df = train_val_frames(df_img)
train_df = train_full  # alias for older cells
print("train (full):", len(train_full), "| val:", len(val_df))
print(train_full.groupby(["dataset", "split_study"]).size())

In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from tqdm.auto import tqdm

RANDOM_SEED = 42
BATCH_SIZE = 64  # lower to 32 if CUDA OOM
EPOCHS = 3
LR = 1e-4
NUM_WORKERS = 0  # Colab/Jupyter: must be 0
USE_AMP = True
SAVE_CHECKPOINT_EACH_EPOCH = True
RUN_ID = "image_resnet18_baseline"

torch.manual_seed(RANDOM_SEED)
assert torch.cuda.is_available(), "Enable GPU runtime before training (see first cell)."
device = torch.device("cuda")
print("device:", device, "|", torch.cuda.get_device_name(0))

# Always reload splits here (needs SETUP cell: PROJECT_ROOT + images unzipped)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "training" / "src"))
from cohort_image import load_image_cohort, train_val_frames

if "df_img" not in globals():
    df_img = load_image_cohort(PROJECT_ROOT)
train_full, val_df = train_val_frames(df_img)
print("Reloaded splits | train:", len(train_full), "| val:", len(val_df))

train_df = train_full
print("FULL cohort train rows:", len(train_df), "| epochs:", EPOCHS)
if len(train_df) < 40_000:
    raise ValueError(
        f"Expected ~44,090 train rows, got {len(train_df)}. "
        "Re-run the LOAD cell above (defines train_full), then this cell."
    )
print("~batches/epoch:", (len(train_df) + BATCH_SIZE - 1) // BATCH_SIZE)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CohortImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, root: Path, transform):
        self.frame = frame.reset_index(drop=True)
        self.root = root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        path = self.root / row["cohort_image_local_path"]
        img = Image.open(path).convert("RGB")
        x = self.transform(img)
        y = int(row["label_binary"])
        return x, y


train_ds = CohortImageDataset(train_df, PROJECT_ROOT, train_tf)
val_ds = CohortImageDataset(val_df, PROJECT_ROOT, val_tf)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

weights = compute_class_weight(
    "balanced", classes=np.array([0, 1]), y=train_df["label_binary"].to_numpy()
)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and device.type == "cuda")

ckpt_dir = PROJECT_ROOT / "runs" / RUN_ID
ckpt_dir.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for xb, yb in tqdm(train_loader, desc=f"epoch {epoch + 1} train"):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP and device.type == "cuda"):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += float(loss.item()) * len(yb)
    print(f"epoch {epoch + 1} train loss:", running / len(train_ds))
    if SAVE_CHECKPOINT_EACH_EPOCH:
        p = ckpt_dir / f"resnet18_epoch{epoch + 1}.pt"
        torch.save({"epoch": epoch + 1, "model": model.state_dict()}, p)
        print("checkpoint:", p)
train_seconds = time.perf_counter() - t0
print(f"Training finished in {train_seconds / 60:.1f} min")

## Validation metrics and curves

Threshold-free curves use **`score_fake`** (softmax probability for class 1). Default **0.5** threshold gives accuracy / macro-F1 / the classification report below.

Artifacts save under `runs/image_resnet18_baseline/` on the Colab project root; the last cell can download JSON and PNGs to your Mac.

In [ ]:
import json

import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

RUN_DIR = PROJECT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

model.eval()
all_y, all_pred, all_prob = [], [], []
t0 = time.perf_counter()
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        pred = logits.argmax(dim=1).cpu().numpy()
        all_y.extend(yb.numpy().tolist())
        all_pred.extend(pred.tolist())
        all_prob.extend(prob.tolist())
infer_seconds = time.perf_counter() - t0

y_val = np.array(all_y)
y_pred = np.array(all_pred)
score_fake = np.array(all_prob)

acc = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro")
roc_auc = roc_auc_score(y_val, score_fake)
avg_precision = average_precision_score(y_val, score_fake)

print("Validation metrics (pooled val, threshold=0.5):")
print(f"  accuracy:  {acc:.4f}")
print(f"  macro_f1:  {macro_f1:.4f}")
print(f"  roc_auc:   {roc_auc:.4f}")
print(f"  avg_prec:  {avg_precision:.4f}  (area under PR curve)")
print(f"  infer_s:   {infer_seconds:.1f}")
print()
print(classification_report(y_val, y_pred, target_names=["real (0)", "fake (1)"]))

# --- confusion matrix ---
cm = confusion_matrix(y_val, y_pred, labels=[0, 1])
fig_cm, ax_cm = plt.subplots(figsize=(4, 3.5))
im = ax_cm.imshow(cm, cmap="Blues")
ax_cm.set_xticks([0, 1], labels=["pred real", "pred fake"])
ax_cm.set_yticks([0, 1], labels=["true real", "true fake"])
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")
ax_cm.set_title("ResNet-18 — confusion matrix (validation)")
fig_cm.colorbar(im, ax=ax_cm, fraction=0.046)
plt.tight_layout()
plt.show()

# --- ROC + precision-recall ---
fpr, tpr, _ = roc_curve(y_val, score_fake)
prec, rec, _ = precision_recall_curve(y_val, score_fake)

fig_curves, axes = plt.subplots(1, 2, figsize=(9, 3.8))

axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curve (fake = positive)")
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, lw=2, label=f"AP = {avg_precision:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–recall curve (fake = positive)")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

pred_df = val_df[["sample_id", "dataset"]].copy()
pred_df["y_true"] = y_val
pred_df["y_pred"] = y_pred
pred_df["score_fake"] = score_fake
display(pred_df.head())

metrics = {
    "run_id": RUN_ID,
    "model": "resnet18_imagenet_finetune",
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "accuracy": float(acc),
    "macro_f1": float(macro_f1),
    "f1_real": float(f1_score(y_val, y_pred, pos_label=0)),
    "f1_fake": float(f1_score(y_val, y_pred, pos_label=1)),
    "roc_auc": float(roc_auc),
    "average_precision": float(avg_precision),
    "train_seconds": round(train_seconds, 4),
    "infer_seconds_val": round(infer_seconds, 4),
    "split_policy": "fakeddit_official_plus_fnn_80_20_seed_42",
    "random_seed": RANDOM_SEED,
    "platform": "google_colab_t4",
    "source_notebook": "training/training_image_resnet.ipynb",
}

pred_path = RUN_DIR / "predictions_val.tsv"
pred_df.to_csv(pred_path, sep="\t", index=False)

metrics_path = RUN_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

cm_path = RUN_DIR / "confusion_matrix.png"
curves_path = RUN_DIR / "roc_pr_curves.png"
fig_cm.savefig(cm_path, dpi=150, bbox_inches="tight")
fig_curves.savefig(curves_path, dpi=150, bbox_inches="tight")

torch.save(model.state_dict(), RUN_DIR / "resnet18_state.pt")

from colab_setup import persist_run_to_drive

persist_run_to_drive(RUN_DIR)

print("Saved:", metrics_path, pred_path, cm_path, curves_path)
print("Checkpoints persisted to My Drive/runs/ — fusion notebook will sync automatically.")
print(json.dumps(metrics, indent=2))